# AirNow and CAM-chem

First we need to import the driver.

In [1]:
from melodies_monet import driver

In [2]:
# Needed if you want to make changes to `melodies_monet` and don't want to restart kernel:
%load_ext autoreload

%autoreload 2

## Initiate the analysis class

Now lets create an instance of the {mod}`melodies_monet.driver` {class}`~melodies_monet.driver.analysis` class.
It consists of 4 main parts: model instances, observation instances, a paired instance of both.
This will allow us to move things around the plotting function for spatial and overlays and more complex plots.

In [3]:
an = driver.analysis()
an

analysis(
    control='control.yaml',
    control_dict=None,
    models={},
    obs={},
    paired={},
    start_time=None,
    end_time=None,
    time_intervals=None,
    download_maps=True,
    output_dir=None,
    output_dir_save=None,
    output_dir_read=None,
    debug=False,
    save=None,
    read=None,
    regrid=False,
)

## Control File

Read in the required yaml control file that sets up all the definitions of what we want to pair and plot.

In [4]:
an.control = 'control_camchem.yaml'
an.read_control()
an.control_dict

{'analysis': {'start_time': '2019-09-01-00:00:00',
  'end_time': '2019-09-09-00:00:00',
  'output_dir': './output/camchem',
  'download_maps': False,
  'debug': False},
 'model': {'cam-chem': {'files': 'example:camchem:fv',
   'mod_type': 'cesm_fv',
   'radius_of_influence': 150000.0,
   'mapping': {'airnow': {'O3': 'OZONE'}},
   'projection': None,
   'plot_kwargs': {'color': 'dodgerblue', 'marker': '+', 'linestyle': '-.'}}},
 'obs': {'airnow': {'use_airnow': True,
   'filename': 'example:airnow:2019-09',
   'obs_type': 'pt_sfc',
   'variables': {'PM2.5': {'unit_scale': 1,
     'unit_scale_method': '*',
     'nan_value': -1.0,
     'ylabel_plot': 'PM2.5 (ug/m3)',
     'ty_scale': 2.0,
     'vmin_plot': 0.0,
     'vmax_plot': 22.0,
     'vdiff_plot': 15.0,
     'nlevels_plot': 23},
    'OZONE': {'unit_scale': 1,
     'unit_scale_method': '*',
     'nan_value': -1.0,
     'ylabel_plot': 'Ozone (ppbv)',
     'vmin_plot': 15.0,
     'vmax_plot': 55.0,
     'vdiff_plot': 20.0,
     'nlevel

## Load the model data 

The driver will automatically loop through the "models" found in the model section of the control file and create model classes for each. Classes include the label, mapping information, and xarray object as well as the filenames.  Note it can open multiple files easily by including wildcards. Here we are only opening one CAM-chem file.

In [5]:
an.open_models()

cesm_fv
example:camchem:fv
**** Reading CESM FV model output...


In [6]:
an.models

{'cam-chem': model(
     model='cesm_fv',
     is_global=False,
     radius_of_influence=150000.0,
     mod_kwargs={'var_list': ['O3']},
     file_str='example:camchem:fv',
     label='cam-chem',
     obj=...,
     extra_calc=None,
     mapping={'airnow': {'O3': 'OZONE'}},
     variable_dict=None,
     label='cam-chem',
     ...
 )}

In [7]:
an.models['cam-chem'].obj

<xarray.Dataset> Size: 9MB
Dimensions:    (time: 36, z: 1, y: 192, x: 288)
Coordinates:
  * lev        (z) float64 8B 992.5
  * time       (time) datetime64[ns] 288B 2019-09-01T06:00:00 ... 2019-09-10
    longitude  (y, x) float64 442kB -180.0 -178.8 -177.5 ... 176.2 177.5 178.8
    latitude   (y, x) float64 442kB -90.0 -90.0 -90.0 -90.0 ... 90.0 90.0 90.0
Dimensions without coordinates: z, y, x
Data variables:
    O3         (time, z, y, x) float32 8MB dask.array<chunksize=(18, 1, 192, 288), meta=np.ndarray>
Attributes:
    Conventions:       CF-1.0
    source:            CAM
    case:              fmerra.2.1003.FCSD.f09.qfedcmip.56L.001.branch02
    logname:           buchholz
    host:              cheyenne3
    initial_file:      /glade/p/cesmdata/cseg/inputdata/atm/cam/inic/fv/f.e20...
    topography_file:   /glade/p/cesmdata/cseg/inputdata/atm/cam/met/MERRA2/0....
    model_doi_url:     https://doi.org/10.5065/D67H1H0V
    time_period_freq:  hour_6
    history:           Mon Feb 28 16:25:23 2022: ncks -7 -L 1 --baa=4 --ppc d...
    NCO:               netCDF Operators version 5.0.6 (Homepage = http://nco....

In [8]:
# All the info in the model class can be called here.
print(an.models['cam-chem'].label)
print(an.models['cam-chem'].mapping)

cam-chem
{'airnow': {'O3': 'OZONE'}}


In [9]:
# All the info in the analysis class can also be called.
print(an.start_time)
print(an.end_time)
print(an.download_maps)

2019-09-01 00:00:00
2019-09-09 00:00:00
True


## Open Obs

Now for monet-analysis we will open preprocessed data in either netcdf icartt or some other format.  We will not be retrieving data like monetio does for some observations (ie aeronet, airnow, etc....).  Instead we will provide utitilies to do this so that users can add more data easily.

Like models we list all obs objects in the yaml file and it will loop through and create driver.observation instances that include the model type, file, objects (i.e. data object) and label  

In [10]:
an.control_dict['obs']

{'airnow': {'use_airnow': True,
  'filename': 'example:airnow:2019-09',
  'obs_type': 'pt_sfc',
  'variables': {'PM2.5': {'unit_scale': 1,
    'unit_scale_method': '*',
    'nan_value': -1.0,
    'ylabel_plot': 'PM2.5 (ug/m3)',
    'ty_scale': 2.0,
    'vmin_plot': 0.0,
    'vmax_plot': 22.0,
    'vdiff_plot': 15.0,
    'nlevels_plot': 23},
   'OZONE': {'unit_scale': 1,
    'unit_scale_method': '*',
    'nan_value': -1.0,
    'ylabel_plot': 'Ozone (ppbv)',
    'vmin_plot': 15.0,
    'vmax_plot': 55.0,
    'vdiff_plot': 20.0,
    'nlevels_plot': 21}}}}

In [11]:
an.open_obs()

In [12]:
# All the info in the observation class can also be called.
an.obs['airnow'].obj

<xarray.Dataset> Size: 1GB
Dimensions:     (x: 3786, time: 2091, y: 1)
Coordinates:
  * x           (x) int64 30kB 0 1 2 3 4 5 6 ... 3780 3781 3782 3783 3784 3785
  * time        (time) datetime64[ns] 17kB 2019-09-01 ... 2019-09-30T00:30:00
    latitude    (y, x) float64 30kB ...
    longitude   (y, x) float64 30kB ...
    siteid      (y, x) <U12 182kB ...
Dimensions without coordinates: y
Data variables: (12/30)
    BARPR       (time, y, x) float64 63MB ...
    BC          (time, y, x) float64 63MB ...
    CO          (time, y, x) float64 63MB ...
    NO          (time, y, x) float64 63MB ...
    NO2         (time, y, x) float64 63MB ...
    NO2Y        (time, y, x) float64 63MB ...
    ...          ...
    cmsa_name   (y, x) float64 30kB ...
    msa_code    (y, x) float64 30kB ...
    msa_name    (y, x) <U52 787kB ...
    state_name  (y, x) <U2 30kB ...
    epa_region  (y, x) <U5 76kB ...
    time_local  (time, y, x) datetime64[ns] 63MB ...
Attributes:
    title:         
    format:        NetCDF-4
    date_created:  2021-06-07

## Pair model and obs data

In [ ]:
%%time

an.pair_data()

1, in pair data


In [ ]:
an.paired

In [ ]:
an.paired['airnow_cam-chem'].obj

## Generate plots

In [ ]:
%%time

an.plotting()

**10 Figures**

::::{card-carousel} 10

:::{card} Figure 1
:img-background: output/camchem/plot_grp1.timeseries.OZONE.2019-09-01_00.2019-09-09_00.all.CONUS.png
:width: 50%
:::

:::{card} Figure 2
:img-background: output/camchem/plot_grp1.timeseries.OZONE.2019-09-01_00.2019-09-09_00.epa_region.R1.png
:width: 50%
:::

:::{card} Figure 3
:img-background: output/camchem/plot_grp2.taylor.OZONE.2019-09-01_00.2019-09-09_00.all.CONUS.png
:width: 50%
:::

:::{card} Figure 4
:img-background: output/camchem/plot_grp2.taylor.OZONE.2019-09-01_00.2019-09-09_00.epa_region.R1.png
:width: 50%
:::

:::{card} Figure 5
:img-background: output/camchem/plot_grp3.spatial_bias.OZONE.2019-09-01_00.2019-09-09_00.all.CONUS.airnow_cam-chem.png
:width: 50%
:::

:::{card} Figure 6
:img-background: output/camchem/plot_grp3.spatial_bias.OZONE.2019-09-01_00.2019-09-09_00.epa_region.R1.airnow_cam-chem.png
:width: 50%
:::

:::{card} Figure 7
:img-background: output/camchem/plot_grp4.spatial_overlay.OZONE.2019-09-01_00.2019-09-09_00.all.CONUS.airnow_cam-chem.png
:width: 50%
:::

:::{card} Figure 8
:img-background: output/camchem/plot_grp4.spatial_overlay.OZONE.2019-09-01_00.2019-09-09_00.epa_region.R1.airnow_cam-chem.png
:width: 50%
:::

:::{card} Figure 9
:img-background: output/camchem/plot_grp5.boxplot.OZONE.2019-09-01_00.2019-09-09_00.all.CONUS.png
:width: 50%
:::

:::{card} Figure 10
:img-background: output/camchem/plot_grp5.boxplot.OZONE.2019-09-01_00.2019-09-09_00.epa_region.R1.png
:width: 50%
:::

::::